# MiniGrid baselines on Kaggle

Select **GPU**, enable **Internet**, and attach the updated baseline source via **Add Input**. The notebook generates training levels and runs one method and seed all the way to 30,000 actual PPO batches. Set `METHOD` to `dr`, `plr`, `plr_perp`, or `accel`.

The trainer saves reward and interaction data every main-loop batch. It evaluates the live policy on OOD layouts every 500 loop batches by default and at the end of training. CV scores are collected from frozen policies at five PPO milestones for methods that score levels. The notebook writes the training CSV, raw OOD episodes, CV samples, and final aggregates. The seven reference layouts work by default. Set `OOD_LAYOUTS_JSON` to the verified extra grids for the full paper suite.

Train with **Save & Run All**. This workflow writes no checkpoints or model files. Restarting an interrupted run starts that seed again. To combine completed reports, switch to `MODE = 'aggregate'` and set `FINAL_OOD_REPORTS`. Aggregation can run on CPU.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path
from time import perf_counter
from textwrap import dedent

source_start = perf_counter()

MODE = 'train'
METHOD = 'dr'  # dr, plr, plr_perp, accel
SEED = 0
OOD_LAYOUTS_JSON = ''  # Optional asset file with the five additional paper grids.
OOD_EVERY = 500
OOD_CURVE_EPISODES = 20
OOD_FINAL_EPISODES = 100
CV_LEVELS = ['16Rooms', 'Labyrinth', 'Labyrinth2', 'Maze', 'Maze2']
CV_LAYOUTS_JSON = ''
FINAL_OOD_REPORTS = []  # Aggregate mode accepts completed PACE and baseline reports.

if MODE not in ('train', 'aggregate'):
    raise ValueError('Set MODE to train or aggregate.')
if METHOD not in ('dr', 'plr', 'plr_perp', 'accel'):
    raise ValueError('Select dr, plr, plr_perp, or accel.')

WORK = Path('/kaggle/working')
runs = WORK / 'pace'
runs.mkdir(parents=True, exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix=f'{METHOD}-seed{SEED}-{MODE}-', dir=runs))
print(f'Session output: {RUN}', flush=True)
SOURCE = ''  # Optional exact path to the extracted dataset folder or source ZIP.
DATASET_SLUG = 'pace-minigrid-baseline-source-90590f8'
SOURCE_ZIP = 'PACE-Research-kaggle-baselines.zip'
required_files = ('scripts/setup_kaggle.sh', 'train/maze_baselines.py', 'train/pace_minigrid.py')

def is_current_source(path):
    if path.is_dir():
        return (all((path / name).is_file() for name in required_files)
                and b'--final-only' in (path / 'train/pace_minigrid.py').read_bytes())
    if path.is_file() and path.suffix.lower() == '.zip':
        try:
            with zipfile.ZipFile(path) as archive:
                return (all(name in archive.namelist() for name in required_files)
                        and b'--final-only' in archive.read('train/pace_minigrid.py'))
        except zipfile.BadZipFile:
            return False
    return False

def find_source(inputs):
    # Check both mount layouts directly. Kaggle input directories can be symlinks.
    roots = [inputs, *inputs.glob('*'), *inputs.glob('datasets/*/*'),
             *inputs.glob('datasets/*/*/versions/*')]
    candidates = [
        path for root in roots for path in (root, root / SOURCE_ZIP)
        if is_current_source(path)
    ]
    preferred = [path for path in candidates if DATASET_SLUG in path.parts]
    candidates = preferred or candidates
    # A dataset can expose both its extracted tree and the ZIP it came from.
    candidates = [path for path in candidates if path.is_dir()] or candidates
    unique = {path.resolve(): path for path in sorted(candidates)}
    if not unique:
        raise FileNotFoundError(
            f'Attach the latest {DATASET_SLUG} dataset via Add Input, or set SOURCE to its exact path.'
        )
    if len(unique) > 1:
        options = ', '.join(str(path) for path in unique.values())
        raise RuntimeError(f'Multiple valid sources: {options}. Set SOURCE to one of these paths.')
    return next(iter(unique.values()))

source = Path(SOURCE) if SOURCE else find_source(Path('/kaggle/input'))
if SOURCE and not is_current_source(source):
    raise ValueError(f'SOURCE must point to the latest extracted baseline code or ZIP: {source}')
print(f'Using baseline source: {source}', flush=True)

REPO = Path(tempfile.mkdtemp(prefix='pace-baselines-source-'))
RUNTIME = Path('/tmp/pace-kaggle')
if source.is_dir():
    shutil.copytree(source, REPO, dirs_exist_ok=True)
else:
    with zipfile.ZipFile(source) as archive:
        archive.extractall(REPO)

if MODE == 'aggregate':
    env = {**os.environ, 'PACE_RUNTIME_DIR': str(RUNTIME), 'JAX_PLATFORMS': 'cpu',
           'MPLBACKEND': 'Agg', 'PYTHONUNBUFFERED': '1'}
else:
    # Use Kaggle's unmodified driver environment for this diagnostic.
    smi = shutil.which('nvidia-smi')
    if not smi or subprocess.run([smi, '-L'], check=False).returncode != 0:
        print('nvidia-smi did not list a GPU. Checking the CUDA driver directly.')

    visible_device = os.environ.get('CUDA_VISIBLE_DEVICES', '0').split(',', 1)[0]
    env = {**os.environ, 'PACE_RUNTIME_DIR': str(RUNTIME), 'CUDA_VISIBLE_DEVICES': visible_device,
           'JAX_PLATFORMS': 'cuda', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
           'MPLBACKEND': 'Agg', 'PYTHONUNBUFFERED': '1'}
    # Keep mounted driver libraries while letting the wheels provide the CUDA toolkit.
    paths = ['/usr/local/nvidia/lib64', *os.environ.get('LD_LIBRARY_PATH', '').split(os.pathsep)]
    driver_paths = dict.fromkeys(path for path in paths if Path(path).is_absolute()
                                and Path(path).name not in {'compat', 'stubs'}
                                and (Path(path) / 'libcuda.so.1').is_file())
    if driver_paths:
        env['LD_LIBRARY_PATH'] = os.pathsep.join(driver_paths)
    else:
        env.pop('LD_LIBRARY_PATH', None)

    driver_probe = dedent('''
    import ctypes
    cuda = ctypes.CDLL('libcuda.so.1')
    status = cuda.cuInit(0)
    if status != 0:
        raise RuntimeError(f'CUDA driver initialization failed (cuInit={status})')
    count = ctypes.c_int()
    status = cuda.cuDeviceGetCount(ctypes.byref(count))
    if status != 0 or count.value < 1:
        raise RuntimeError(f'No visible CUDA device (status={status}, count={count.value})')
    ''')
    try:
        subprocess.run([sys.executable, '-c', driver_probe], env=env, check=True)
    except subprocess.CalledProcessError:
        raise RuntimeError('Kaggle has no usable CUDA device for this run. Select a GPU accelerator, '
                           'then restart the session or save a new GPU-enabled version.') from None

source_seconds = perf_counter() - source_start

In [ ]:
def run_logged(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'Log: {log_path}', flush=True)
    with log_path.open('w', encoding='utf-8') as log, subprocess.Popen(
        command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace',
    ) as process:
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end='', flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f'{log_path.stem} failed with exit code {code}. See {log_path}')

setup_start = perf_counter()
run_logged(['bash', 'scripts/setup_kaggle.sh'], RUN / 'setup.log')
PACE_PYTHON = str(RUNTIME / 'venv/bin/python')

probe = '''
import importlib.metadata as metadata
import json
import jax
import jax.numpy as jnp
from jaxued.environments import Maze
from train.pace_minigrid import MiniGridConfig
config = MiniGridConfig.baseline('dr')
assert config.method == 'dr' and config.score_type is None
from train.pace_analysis import MiniGridAnalysis
devices = jax.local_devices(backend='gpu')
assert len(devices) == 1, devices
result = jax.jit(lambda x: x @ x)(jnp.ones((16, 16))).block_until_ready()
assert result.device.platform == 'gpu' and float(result[0, 0]) == 16.0
pin = '0f8f1284677375b889e4f13a32c9617cd009f8c4'
origin = json.loads(metadata.distribution('jaxued').read_text('direct_url.json'))
assert origin['vcs_info']['commit_id'] == pin
print({'device': str(devices[0]), 'jax': jax.__version__, 'jaxued_commit': pin, 'environment': Maze.__name__})
'''
if MODE == 'train':
    run_logged([PACE_PYTHON, '-c', probe], RUN / 'gpu-probe.log')
setup_seconds = source_seconds + perf_counter() - setup_start

In [ ]:
import json

if MODE == 'train':
    suite = 'paper' if OOD_LAYOUTS_JSON else 'reference'
    command = [
        PACE_PYTHON, '-m', 'train.pace_minigrid', 'train', '--device', 'gpu',
        '--method', METHOD, '--seed', str(SEED), '--budget', 'ppo', '--final-only',
        '--suite', suite, '--ood-every', str(OOD_EVERY),
        '--curve-episodes', str(OOD_CURVE_EPISODES), '--final-episodes', str(OOD_FINAL_EPISODES),
        '--cv-levels', *CV_LEVELS, '--output-dir', str(RUN),
    ]
    if OOD_LAYOUTS_JSON:
        command += ['--levels-json', OOD_LAYOUTS_JSON]
    if CV_LAYOUTS_JSON:
        command += ['--cv-levels-json', CV_LAYOUTS_JSON]
    run_logged(command, RUN / 'train.log')
    report_path = RUN / 'run_summary.json'
    summary = json.loads(report_path.read_text(encoding='utf-8'))
    if summary['status'] != 'training_complete' or summary['counts']['ppo_batches'] != 30_000:
        raise RuntimeError(f'Run did not reach the expected PPO budget. See {report_path}')
    summary['timing']['setup_seconds'] = round(setup_seconds, 2)
    report_path.write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(summary, indent=2))

In [ ]:
reports = [str(RUN / 'final_ood.json')] if MODE == 'train' else FINAL_OOD_REPORTS
if not reports:
    raise ValueError('Attach completed evaluation outputs and fill FINAL_OOD_REPORTS.')
run_logged([PACE_PYTHON, '-m', 'train.pace_analysis', 'aggregate',
            '--reports', *reports, '--output-dir', str(RUN)], RUN / 'aggregate.log')
print((RUN / 'ood_aggregate.json').read_text(encoding='utf-8'))